# Final evaluation and benchmarks (Phase 7)

Before running: **Settings > Accelerator > GPU T4 x2**, and **Settings > Internet > On**.

This is the one sealed test-split evaluation, run from the frozen `final-config` tag. It logs each test access. Run it once.
Download `final_results.zip` from the Output panel when the last cell finishes. `eval_results.zip` appears earlier as a safety copy.

In [ ]:
!nvidia-smi

In [ ]:
import os
REPO = 'https://github.com/FaatehHaneef/Clinical-records-screener.git'
WORK = '/kaggle/temp/Clinical-records-screener'
if not os.path.exists(WORK):
    !git clone -q --branch final-config {REPO} {WORK}
%cd {WORK}
!git rev-parse HEAD
!git describe --tags

In [ ]:
!pip install -q -r requirements.txt 2>&1 | tail -3
!python -c "import torch, transformers, peft; print(torch.__version__, torch.cuda.is_available(), transformers.__version__, peft.__version__)"

In [ ]:
!python scripts/fetch_resources.py --models-from-lock
!python scripts/fetch_adapters.py

In [ ]:
!python -m pytest -q tests/test_split.py tests/test_metrics.py tests/test_verifier.py tests/test_decoders.py tests/test_attention.py tests/test_speculative.py

In [ ]:
E = 'python -m screener.eval.evaluate --batch-size 32'
BASE = '--model models/qwen2.5-1.5b'
AFTER = '--model models/qwen2.5-1.5b --adapter models/crs-adapter'
!{E} {BASE} --split validation --output results/eval/before_validation
!{E} {AFTER} --split validation --output results/eval/after_validation
!{E} --baseline --split validation --output results/eval/baseline_validation

In [ ]:
!{E} {BASE} --split test --final --output results/eval/before_test
!{E} {AFTER} --split test --final --output results/eval/after_test
!{E} --baseline --split test --final --output results/eval/baseline_test
!cat results/test_access_log.jsonl

In [ ]:
import shutil
shutil.make_archive('/kaggle/working/eval_results', 'zip', 'results/eval')
shutil.copy('results/test_access_log.jsonl', '/kaggle/working/test_access_log.jsonl')
print('Safety copy of the evaluation saved')

In [ ]:
!python benchmarks/bench_cache.py --model models/qwen2.5-1.5b --adapter models/crs-adapter --label kaggle-t4-finetuned

In [ ]:
!python benchmarks/bench_attention.py --label kaggle-t4

In [ ]:
!python benchmarks/bench_speculative.py --target models/qwen2.5-1.5b --target-adapter models/crs-adapter --draft models/qwen2.5-0.5b --draft-adapter models/crs-draft-adapter --label kaggle-t4-finetuned-draft
!python benchmarks/bench_speculative.py --target models/qwen2.5-1.5b --target-adapter models/crs-adapter --draft models/qwen2.5-0.5b --label kaggle-t4-base-draft

In [ ]:
import glob, json
target = '/kaggle/working/final_results'
shutil.rmtree(target, ignore_errors=True)
shutil.copytree('results/eval', f'{target}/eval')
shutil.copy('results/test_access_log.jsonl', target)
for path in glob.glob('results/bench_*kaggle-t4*.json'):
    shutil.copy(path, target)
shutil.make_archive('/kaggle/working/final_results', 'zip', target)
for name in ('before_validation', 'after_validation', 'before_test', 'after_test', 'baseline_test'):
    overall = json.load(open(f'results/eval/{name}/metrics.json'))['metrics']['classification']['overall']
    print(name, round(overall['accuracy'], 4), round(overall['macro_f1'], 4))
print('Download final_results.zip from the Output panel')